List All Imports

In [1]:
import os
from chess import pgn
from tqdm import tqdm
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from chess import Board
import tensorflow as tf
import pickle
import json

Load raw games and preprocess them for data set

In [2]:
# get the game files
files = [file for file in os.listdir("new_training_data") if file.endswith(".pgn")]

In [3]:
# load the games
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
            
    return games

In [4]:
# write all the games together 

games = []
for file in tqdm(files):
    games.extend(load_pgn(f"new_training_data/{file}"))

100%|██████████| 15/15 [00:45<00:00,  3.03s/it]


In [5]:
len(games) # check how many 

15000

In [6]:
#translating the board into a matrix for the predition process 
def board_to_matrix(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix

# create the inputs based off game position and next move 
def create_input_for_nn(games):
    X = []
    y = []

    for game in games:
        board = game.board()
        ply_count = 0  # counts half-moves

        for move in game.mainline_moves():
            if ply_count >= 10: # skip the first 5 moves from each side
                X.append(board_to_matrix(board))
                y.append(move.uci())

            board.push(move)
            ply_count += 1

    return X, y

# encode all the moves
def encode_moves(moves):
    # Load old encoding
    with open("models/simulated_filtered_model/move_to_int.pkl", "rb") as f:
        move_to_int_old = pickle.load(f)

    # Find new unique moves from input moves
    new_moves = set(moves) - set(move_to_int_old.keys())

    # Start indexing new moves after the max index in old mapping
    max_index = max(move_to_int_old.values())

    # Create a copy of old mapping to extend
    move_to_int = dict(move_to_int_old)

    # Add new moves with new indices
    for i, move in enumerate(sorted(new_moves), start=max_index + 1):
        move_to_int[move] = i

    # Encode the moves using the extended mapping
    encoded = [move_to_int[move] for move in moves]

    return encoded, move_to_int

In [7]:
# create the training data
X, y = create_input_for_nn(games)
y, move_to_int = encode_moves(y)
y = tf.keras.utils.to_categorical(y, num_classes=len(move_to_int))
X = np.array(X)

Load Pretrained model, freeze convolutional layers and create new sequence of dense layers for transfer learning

In [8]:
pretrained_model = tf.keras.models.load_model("models/simulated_filtered_model/SSMF_50EPOCHS.keras")

print("Layers in pretrained model:")
for i, layer in enumerate(pretrained_model.layers):
    print(f"{i}: {layer.name}")

conv_base = tf.keras.Sequential(pretrained_model.layers[:3])

conv_base.build((None, 8, 8, 12)) 

conv_base.trainable = False

new_model = tf.keras.Sequential([
        conv_base,
        tf.keras.layers.Dense(2048, activation='relu', name='new_dense1'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1024, activation='relu', name='new_dense2'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(len(move_to_int), activation='softmax', name='new_output')
    ])

Layers in pretrained model:
0: conv2d_2
1: conv2d_3
2: flatten_1
3: dense_2
4: dense_3


Compile and train new layers in model

In [9]:
new_model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
new_model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

new_model.fit(X, y, epochs=50, validation_split=0.1, batch_size=64, callbacks=callbacks)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 2048)           │        80,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_dense1 (Dense)              │ (None, 2048)           │     4,196,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_dense2 (Dense)              │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_output (Dense)              │ (None, 1966)           │     2,015,150 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,390,510 (32.01 MB)

 Trainable params: 8,309,678 (31.70 MB)

 Non-trainable params: 80,832 (315.75 KB)

Epoch 1/50
26810/26810 ━━━━━━━━━━━━━━━━━━━━ 396s 15ms/step - accuracy: 0.0419 - loss: 5.5350 - val_accuracy: 0.0608 - val_loss: 4.7004
Epoch 2/50
26810/26810 ━━━━━━━━━━━━━━━━━━━━ 384s 14ms/step - accuracy: 0.0592 - loss: 4.7374 - val_accuracy: 0.0685 - val_loss: 4.4417
Epoch 3/50
26810/26810 ━━━━━━━━━━━━━━━━━━━━ 368s 14ms/step - accuracy: 0.0660 - loss: 4.5470 - val_accuracy: 0.0724 - val_loss: 4.3294
Epoch 4/50
26810/26810 ━━━━━━━━━━━━━━━━━━━━ 388s 14ms/step - accuracy: 0.0716 - loss: 4.4389 - val_accuracy: 0.0751 - val_loss: 4.2671
Epoch 5/50
26810/26810 ━━━━━━━━━━━━━━━━━━━━ 392s 15ms/step - accuracy: 0.0759 - loss: 4.3658 - val_accuracy: 0.0769 - val_loss: 4.2228
Epoch 6/50
26810/26810 ━━━━━━━━━━━━━━━━━━━━ 383s 14ms/step - accuracy: 0.0793 - loss: 4.3095 - val_accuracy: 0.0786 - val_loss: 4.1980
Epoch 7/50
26810/26810 ━━━━━━━━━━━━━━━━━━━━ 391s 15ms/step - accuracy: 0.0831 - loss: 4.2644 - val_accuracy: 0.0801 - val_loss: 4.1742
Epoch 8/50
26810/26810 ━━━━━━━━━━━━━━━━━━━━ 384s 14ms/s

Save model and related data to training and configuration

In [10]:
# save the model
new_model.save("models/TL_LateGame/TL_50EPOCHS.keras")

# save the encoding
with open("models/TL_LateGame/move_to_int.pkl", "wb") as f:
    pickle.dump(move_to_int, f)
int_to_move = {v: k for k, v in move_to_int.items()}
with open("models/TL_LateGame/int_to_move.pkl", "wb") as f:
    pickle.dump(int_to_move, f)
# configuration 
config = {
    "epochs": 50,
    "batch_size": 64 ,
    "validation_split": 0.1,
    "optimizer": "Adam",
    "input_shape": (8, 8, 12),
}
# save the configurations
with open("models/TL_LateGame/train_config.json", "w") as f:
    json.dump(config, f, indent=4)